# Movie Recommendation System Demo

This notebook demonstrates the collaborative filtering recommendation system.

In [ ]:
import sys

sys.path.append("../scripts")
sys.path.append("../")

import polars as pl

# from recommendation.collaborative_filtering import CollaborativeFilteringModel
import json
from pathlib import Path

from recommendation.movielens_cf import MovieLensCF  # noqa: E402
from tmdb_client import TMDBClient  # noqa: E402

In [12]:
@cd ../

SyntaxError: invalid syntax (1738419610.py, line 1)

In [10]:
!ls

1_upload_movie_creds.ipynb    5_replace_mistakes.ipynb
2_add_movie_id.ipynb          6_analytics.ipynb
3_movie_validate.ipynb        7_add_new_movies.ipynb
4_enrich_df_with_params.ipynb 8_recommendation_demo.ipynb


In [34]:
tmdb_client = TMDBClient()

In [3]:
top_n = 5

In [18]:
import os

In [ ]:
os.chdir("..")

In [ ]:
cf_model = MovieLensCF()
sys.path.append("../")
my_movies = pl.read_parquet("data/movies_df.parquet")
my_movies = my_movies.filter(pl.col("omdb_id") != "Not found")

# Generate more candidates since we filter for TMDB posters
cf_recs_raw = cf_model.get_recommendations(my_movies, top_n=top_n * 3)

Loaded MovieLens dataset:
  - 32000204 ratings
  - 87585 movies
  - 200948 users
Mapped 638/667 of your movies to MovieLens
  - 201 liked (rating ~4.5)
  - 437 watched (rating ~3.0)
Found 100 similar users
  Top similar user: ID=175325, overlap=556 movies, score=398.45


In [23]:
cf_recs_raw

[{'title': 'Silence of the Lambs, The',
  'year': 1991,
  'movielens_id': 593,
  'genres': 'Crime|Horror|Thriller',
  'avg_rating': 4.46,
  'num_similar_users': 98,
  'cf_score': 0.97},
 {'title': 'Godfather: Part II, The',
  'year': 1974,
  'movielens_id': 1221,
  'genres': 'Crime|Drama',
  'avg_rating': 4.64,
  'num_similar_users': 95,
  'cf_score': 0.96},
 {'title': '12 Angry Men',
  'year': 1957,
  'movielens_id': 1203,
  'genres': 'Drama',
  'avg_rating': 4.58,
  'num_similar_users': 93,
  'cf_score': 0.94},
 {'title': 'Star Wars: Episode V - The Empire Strikes Back',
  'year': 1980,
  'movielens_id': 1196,
  'genres': 'Action|Adventure|Sci-Fi',
  'avg_rating': 4.52,
  'num_similar_users': 93,
  'cf_score': 0.94},
 {'title': 'Die Hard',
  'year': 1988,
  'movielens_id': 1036,
  'genres': 'Action|Crime|Thriller',
  'avg_rating': 4.28,
  'num_similar_users': 96,
  'cf_score': 0.93},
 {'title': 'Seven (a.k.a. Se7en)',
  'year': 1995,
  'movielens_id': 47,
  'genres': 'Mystery|Thrille

In [26]:
from scripts.generate_recommendations import enrich_cf_recommendations_with_tmdb

In [35]:
# if cf_recs_raw:
# Enrich with TMDB data (required - only movies with posters are included)
tmdb_client = TMDBClient()
recommendations = enrich_cf_recommendations_with_tmdb(cf_recs_raw, tmdb_client)
print(f"\n✓ Generated {len(recommendations)} CF recommendations with TMDB posters")


✓ Generated 0 CF recommendations with TMDB posters


In [ ]:
tmdb_client.movie.search("Silence of the Lambs, The")

TMDbException: Invalid API key: You must be granted a valid key.

In [ ]:
 print(f"TMDB API key loaded: {'Yes' if os.getenv('TMDB_API_KEY') else   'No'}")

In [ ]:
from dotenv import load_dotenv

load_dotenv()

True

In [30]:
rec

{'title': 'Silence of the Lambs, The',
 'year': 1991,
 'movielens_id': 593,
 'genres': 'Crime|Horror|Thriller',
 'avg_rating': 4.46,
 'num_similar_users': 98,
 'cf_score': 0.97}

In [29]:
for rec in cf_recs_raw:
    # Try to find movie on TMDB
    # tmdb_movie = None
    # try:
    tmdb_movie = tmdb_client.get_movie_by_title(rec["title"], rec.get("year"))
    break

TMDbException: Invalid API key: You must be granted a valid key.

In [28]:
recommendations

[]

In [7]:
help(tmdb_client)

Help on TMDBClient in module tmdb_client object:

class TMDBClient(builtins.object)
 |  Client for interacting with The Movie Database (TMDB) API.
 |
 |  Methods defined here:
 |
 |  __init__(self)
 |      Initialize TMDB API client.
 |
 |  discover_by_genres(
 |      self,
 |      genre_ids: List[int],
 |      min_rating: float = 7.0,
 |      limit: int = 20
 |  ) -> List[Dict]
 |      Discover movies by genre IDs with minimum rating.
 |
 |  discover_popular_recent(self, min_year: int = 2020, limit: int = 50) -> List[Dict]
 |      Discover popular recent movies.
 |
 |  enrich_watched_movies_with_tmdb_ids(
 |      self,
 |      movies_df: polars.dataframe.frame.DataFrame
 |  ) -> polars.dataframe.frame.DataFrame
 |      Add TMDB IDs to watched movies dataframe.
 |
 |  get_movie_by_title(self, title: str, year: Optional[int] = None) -> Optional[Dict]
 |      Search for a movie by title and optionally year.
 |
 |  get_recommendations(self, tmdb_id: int, limit: int = 20) -> List[Dict]
 | 

In [9]:
tmdb_client.tmdb.api_key

'nicjah-jomdyx-3pasfI'

## 1. Load Your Movie Data

In [ ]:
movies_df = pl.read_parquet("../data/movies_df.parquet")
print(f"Total movies: {len(movies_df)}")
print(f"Liked movies: {movies_df.filter(pl.col('liked') == True).shape[0]}")
print(
    f"Movies with viewing date: {movies_df.filter(pl.col('viewed').is_not_null()).shape[0]}"
)

movies_df.head()

## 2. Analyze Recent Viewing History (Last 6 Months)

In [ ]:
from datetime import datetime, timedelta

cutoff_date = datetime.now().date() - timedelta(days=180)

recent_movies = movies_df.filter(
    (pl.col("viewed").is_not_null())
    & (pl.col("viewed") >= cutoff_date)
    & (pl.col("liked") == True)
)

print(f"Recent liked movies (last 6 months): {len(recent_movies)}")
recent_movies.sort("viewed", descending=True).head(10)

## 3. Extract Viewing Preferences

In [ ]:
from collections import Counter

genres = []
directors = []
for row in recent_movies.iter_rows(named=True):
    if row["genre"]:
        genres.extend([g.strip() for g in str(row["genre"]).split(",")])
    if row["director"]:
        directors.extend([d.strip() for d in str(row["director"]).split(",")])

print("Top Genres:")
for genre, count in Counter(genres).most_common(10):
    print(f"  {genre}: {count}")

print("\nTop Directors:")
for director, count in Counter(directors).most_common(5):
    print(f"  {director}: {count}")

print(f"\nAverage Rating: {recent_movies['imdb_rating'].mean():.2f}")

## 4. Train the Recommendation Model

In [ ]:
model = CollaborativeFilteringModel(data_path="../data/movies_df.parquet")
model.train(months_back=6)

## 5. Generate Recommendations

In [ ]:
recommendations = model.predict(top_n=5)

print("\n" + "=" * 60)
print("TOP 5 MOVIE RECOMMENDATIONS")
print("=" * 60)

for i, rec in enumerate(recommendations, 1):
    print(f"\n{i}. {rec['title']} ({rec['year']})")
    print(f"   TMDB ID: {rec['tmdb_id']}")
    print(f"   Rating: {rec['rating']:.1f}/10" if rec["rating"] else "   Rating: N/A")
    print(f"   Match Score: {rec['score']}")
    print(f"   Overview: {rec['overview'][:150]}...")
    if rec.get("poster_path"):
        print(f"   Poster: https://image.tmdb.org/t/p/w500{rec['poster_path']}")

## 6. View Saved Recommendations

In [ ]:
latest_rec_path = Path("../data/recommendations/recommendations_latest.json")

if latest_rec_path.exists():
    with open(latest_rec_path, "r") as f:
        saved_recs = json.load(f)

    print(f"Generated at: {saved_recs['generated_at']}")
    print(f"Based on {saved_recs['months_back']} months of viewing history\n")

    rec_df = pl.DataFrame(saved_recs["recommendations"])
    display(rec_df)
else:
    print(
        "No saved recommendations found. Run the generate_recommendations.py script first."
    )

## 7. Visualize Genre Distribution

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

genre_counts = Counter(genres)
top_genres = dict(genre_counts.most_common(10))

plt.figure(figsize=(12, 6))
plt.bar(top_genres.keys(), top_genres.values())
plt.xlabel("Genre")
plt.ylabel("Count")
plt.title("Your Top Genres (Last 6 Months)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. Rating Distribution

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(recent_movies["imdb_rating"].to_list(), bins=20, edgecolor="black")
plt.xlabel("IMDB Rating")
plt.ylabel("Frequency")
plt.title("Rating Distribution of Your Recent Liked Movies")
plt.axvline(
    recent_movies["imdb_rating"].mean(),
    color="red",
    linestyle="--",
    label=f"Mean: {recent_movies['imdb_rating'].mean():.2f}",
)
plt.legend()
plt.tight_layout()
plt.show()